# Bike Explanation Stability

[Open in Colab](https://colab.research.google.com/github/WanruXX/bike_explanation_stability/blob/main/bike_experiment.ipynb)

**Research question:** How representation-dependent is attribution within a correlated feature group, and what information does grouping preserve or conceal?

This notebook runs the representation study (BE1) and fixed-model robustness analysis (BE2) through `main.py`, and displays saved results. BE0 is the historical pilot. The study uses the public UCI Bike Sharing dataset; results are exploratory because TEST was inspected during development.

## 1. Environment Setup

A Colab CPU runtime or local Jupyter environment is sufficient. Setup uses the current source directory or clones the repository into `bike_explanation_stability`. Existing checkouts are reused without pulling.

Dependencies are installed from `requirements-shap.txt`. These version ranges may differ from the exact environment of the recorded runs. Colab storage is temporary; results can be exported using the final cell or written to a persistent `RESULTS_DIR`.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

REPOSITORY_URL = "https://github.com/WanruXX/bike_explanation_stability.git"
CHECKOUT = Path.cwd() / "bike_explanation_stability"

if (Path.cwd() / "main.py").is_file() and (Path.cwd() / "bike_config.py").is_file():
    STUDY_ROOT = Path.cwd().resolve()
else:
    if not CHECKOUT.exists():
        subprocess.run(["git", "clone", REPOSITORY_URL, str(CHECKOUT)], check=True)
    STUDY_ROOT = CHECKOUT.resolve()
if not (STUDY_ROOT / "main.py").is_file():
    raise RuntimeError(f"Study source not found in {STUDY_ROOT}; check the checkout path")
print("Study root:", STUDY_ROOT)
print("Python:", sys.version)
print("Existing checkouts are reused without pulling or overwriting local changes.")

In [ ]:
INSTALL_DEPENDENCIES = True
if INSTALL_DEPENDENCIES:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-r", "requirements-shap.txt"],
        cwd=STUDY_ROOT, check=True,
    )

DATA_DIR = STUDY_ROOT / "data"
RESULTS_DIR = STUDY_ROOT / "results"

def run_cli(*arguments):
    command = [
        sys.executable, str(STUDY_ROOT / "main.py"),
        "--data-dir", str(DATA_DIR), "--results-dir", str(RESULTS_DIR),
        *map(str, arguments),
    ]
    print("Running:", " ".join(command), flush=True)
    subprocess.run(command, cwd=STUDY_ROOT, check=True)

run_cli("--help")

## 2. Study Design

| Stage | Design |
|---|---|
| BE0: pilot | 48 candidate fits and 24 selected models. A chronological-prefix SHAP sampling defect motivated corrected sampling in BE1. Historical stage; not run by this notebook. |
| BE1: representation study | Six TRAIN-only fits: forest and boosting, three temperature representations, seed 42. Candidate index 1 is fixed from BE0 validation selections. |
| BE2: robustness | Fixed BE1 functions and cohorts; seven matched SHAP settings and paired one-day/seven-day block intervals. |

TRAIN is 2011; validation is January-June 2012; TEST is July-December 2012. All representations share rows, targets, and other predictors. Marginal-background SHAP is neither conditional nor causal; summed feature contributions are not a recomputed grouped Shapley game. Findings and limitations are summarized in [README.md](README.md).

## 3. Explicit Data Download And Synthetic Preflight

Downloading is separate from execution, just as in the CLI. Enable `DOWNLOAD_DATA` only when `hour.csv` is absent. Existing data are reused without replacement. The `test` command uses synthetic fixtures, including a small EBM fit; it does not download or fit real-data models.

Global `--data-dir` and `--results-dir` arguments precede the subcommand. `run_cli` preserves that ordering and uses the kernel's Python executable. Experiment commands also run their own preflight checks.

In [ ]:
DOWNLOAD_DATA = False
RUN_PREFLIGHT = True
if DOWNLOAD_DATA:
    run_cli("download")
else:
    print("Download disabled. Enable DOWNLOAD_DATA if hour.csv is absent.")
if RUN_PREFLIGHT:
    run_cli("test")

In [ ]:
DATA_PATH = DATA_DIR / "hour.csv"
print("CSV:", DATA_PATH)
print("Present:", DATA_PATH.is_file())
print("Results:", RESULTS_DIR)
print("Execution never downloads data automatically.")

In [ ]:
RUN_EXPERIMENT = False
WORKFLOW = "all"
BE1_RUN_LABEL = "be1_colab_replay"
BE2_RUN_LABEL = "be2_colab_replay"
SOURCE_RUN_LABEL = "be1_temperature_representation"
REBUILD_MODELS = False
N_JOBS = 1

if WORKFLOW not in {"all", "be1", "be2"}:
    raise ValueError("WORKFLOW must be all, be1 or be2")
print("Workflow:", WORKFLOW, "Enabled:", RUN_EXPERIMENT)
print("Choose fresh output labels for every new attempt, including after failures.")

## 4. Choose A CLI Workflow

- `WORKFLOW="all"`: `python main.py all --be1-run-label ... --be2-run-label ... --n-jobs 1`. BE1 and BE2 run in one process, with live-model reuse and no BE2 reconstruction.
- `WORKFLOW="be1"`: `python main.py be1 --run-label ... --n-jobs 1`. Produces a saved report and predictions, not serialized model files.
- `WORKFLOW="be2"`: `python main.py be2 --source-run-label ... --run-label ... --rebuild-models`. Set `REBUILD_MODELS=True` to authorize fitting saved BE1 settings on TRAIN and verifying predictions. Set `SOURCE_RUN_LABEL` to the report to analyze, including a newly completed BE1 label.

Notebook subprocesses do not retain fitted models in the kernel. Only `all` reuses models across stages here. Both experiment execution and separate BE2 reconstruction are disabled by default. Select a workflow and enable `RUN_EXPERIMENT` in the preceding configuration cell, then run the execution cell below.

### Computation And Reproducibility

BE1 uses 1,024 permutation rows, 20 repeats, 64 SHAP rows, 128 TRAIN background rows and 10 cycles. BE2 uses seven matched cases (36 new SHAP computations for six models) and 1,000 bootstrap repeats. Execution can require substantial CPU time.

Each attempt requires unused output labels. Existing directories, including failed attempts, are protected; resume is not supported. The `all` command checks both output directories before fitting.

Separate BE2 reconstruction requires source artifacts and matching recorded NumPy, SciPy, scikit-learn and SHAP versions. Default Colab packages may differ. A new `all` run produces a BE1/BE2 pair in the same environment.

In [ ]:
if RUN_EXPERIMENT:
    if WORKFLOW == "all":
        run_cli("all", "--be1-run-label", BE1_RUN_LABEL,
                "--be2-run-label", BE2_RUN_LABEL, "--n-jobs", N_JOBS)
    elif WORKFLOW == "be1":
        run_cli("be1", "--run-label", BE1_RUN_LABEL, "--n-jobs", N_JOBS)
    elif WORKFLOW == "be2":
        if not REBUILD_MODELS:
            raise RuntimeError("Separate BE2 requires explicit REBUILD_MODELS=True; use all for live reuse")
        run_cli("be2", "--source-run-label", SOURCE_RUN_LABEL,
                "--run-label", BE2_RUN_LABEL, "--rebuild-models")
    else:
        raise ValueError("WORKFLOW must be all, be1 or be2")
else:
    print("Experiments disabled. Existing reports can be inspected below without training.")

## 5. Inspect Saved BE1 Reports

Set `INSPECT_BE1_LABEL` to a completed run and enable `LOAD_BE1_REPORT`. Inspection is read-only and requires no training. Generated results are excluded from version control, so these cells require a new run or separately supplied artifacts.

Prediction agreement is distinct from explanation agreement. Permutation-repeat standard deviations describe Monte Carlo variation, not sampling confidence intervals. Joint permutation preserves dependence within a group, not between groups.

In [ ]:
import pandas as pd

LOAD_BE1_REPORT = False
INSPECT_BE1_LABEL = SOURCE_RUN_LABEL if WORKFLOW == "be2" else BE1_RUN_LABEL
report = None
record = None
if LOAD_BE1_REPORT:
    report_path = RESULTS_DIR / INSPECT_BE1_LABEL / "report.json"
    report = json.loads(report_path.read_text(encoding="utf-8"))
    print("Loaded:", report_path)
    display(pd.DataFrame([
        {"record_id": item["record_id"],
         "validation_rmse": item["results_validation"]["rmse"],
         "test_rmse": item["results"]["rmse"],
         "test_mae": item["results"]["mae"],
         "near_optimal_on_validation": item["near_optimal_on_validation"]}
        for item in report["records"]
    ]))
    display(pd.DataFrame([
        {"record_id": item["record_id"],
         **item["temperature_attribution"]["individual_mse_increase"],
         "joint_temperature_pfi": item["temperature_attribution"]["joint_mse_increase"]}
        for item in report["records"]
    ]))
    record = report["records"][0]
else:
    print("BE1 report loading disabled.")

In [ ]:
import matplotlib.pyplot as plt

if record is not None:
    figure, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)
    for axis, panel_name in zip(axes, ("individual", "grouped")):
        panels = record["explanations"][panel_name]["panels"]
        ordered = sorted(panels, key=lambda name: panels[name]["mean_mse_increase"])
        axis.barh(ordered, [panels[name]["mean_mse_increase"] for name in ordered])
        axis.axvline(0, color="black", linewidth=0.8)
        axis.set_xlabel("MSE increase after permutation")
        axis.set_title(panel_name.capitalize())
    figure.suptitle(record["record_id"])
    plt.show()
else:
    print("Load a BE1 report before plotting.")

### Local SHAP And Saved Predictions

BE1 explains a reproducible random 64-row subset of its common 1,024-row permutation cohort, sorted for display. The following tables show temporal coverage, signed contributions and saved predictions. Reconstruction error measures additivity, not explanation convergence or causal validity.

In [ ]:
if record is not None and "shap" in record["explanations"]:
    local = record["explanations"]["shap"]
    cohort = pd.DataFrame({"row_id": local["row_ids"], "timestamp": pd.to_datetime(local["timestamps"])})
    print(local["sampling"])
    display(cohort.groupby(cohort["timestamp"].dt.to_period("M")).size().rename("explained_rows"))
    display(pd.DataFrame(local["values"], columns=local["feature_names"], index=local["row_ids"]))
    display(pd.DataFrame(local["group_contributions"], index=local["row_ids"]))
    print("SHAP reconstruction max abs error:", local["reconstruction_max_abs_error"])
else:
    print("Load a BE1 report with SHAP results first.")

In [ ]:
import numpy as np

if report is not None:
    with np.load(RESULTS_DIR / INSPECT_BE1_LABEL / "test_predictions.npz", allow_pickle=False) as saved:
        record_id = report["records"][0]["record_id"]
        display(pd.DataFrame({
            "row_id": saved["row_ids"], "timestamp": saved["timestamps"],
            "target": saved["targets"], "prediction": saved[record_id],
        }).head())
else:
    print("Load a BE1 report before inspecting predictions.")

## 6. Inspect Saved BE2 Robustness Reports

Set `INSPECT_BE2_LABEL` to a completed run and enable `LOAD_BE2_REPORT`. The tables compare within-model SHAP sensitivity, matched cross-model explanations, and paired uncertainty intervals.

SHAP RNG changes measure approximation sensitivity; background changes also alter the empirical reference game. Larger budgets are comparison settings, not evidence of convergence.

One-day/seven-day calendar-block intervals condition on fitted models and the fixed evaluation design. SHAP intervals use occupied blocks of a sparse 64-row cohort. They exclude training uncertainty and do not establish stationarity or statistical equivalence.

In [ ]:
LOAD_BE2_REPORT = False
INSPECT_BE2_LABEL = BE2_RUN_LABEL
GROUP_TO_INSPECT = "temperature"
be2_report = None
if LOAD_BE2_REPORT:
    report_path = RESULTS_DIR / INSPECT_BE2_LABEL / "report.json"
    be2_report = json.loads(report_path.read_text(encoding="utf-8"))
    print("Loaded:", report_path)
    print(be2_report["status"], be2_report["model_modes"])
    display(pd.DataFrame([
        {"record_id": item["record_id"], "case_id": item["case_id"], "axis": item["axis"],
         "mean_abs_temperature_shap": item["temperature"]["mean_absolute_joint_contribution"],
         "temp_absolute_share": item["temperature"]["individual_absolute_share"]["temp"],
         "background_rows": item["background_rows_used"]}
        for item in be2_report["shap_summaries"]
    ]))
else:
    print("BE2 report loading disabled.")

In [ ]:
if be2_report is not None:
    display(pd.DataFrame([
        {"record_id": item["record_id"], "case_id": item["case_id"], "axis": item["axis"],
         "local_group_mae_vs_baseline": item["local_group_agreement"][f"{GROUP_TO_INSPECT}/mae_difference"]}
        for item in be2_report["within_model_sensitivity"]
    ]))
    display(pd.DataFrame([
        {"first": pair["first"], "second": pair["second"], "kind": pair["kind"],
         "case_id": case["case_id"],
         "local_group_mae_between_models": case["local_group_agreement"][f"{GROUP_TO_INSPECT}/mae_difference"]}
        for pair in be2_report["comparisons"] for case in pair["matched_shap_cases"]
    ]))
    interval_rows = []
    for pair in be2_report["comparisons"]:
        for panel in ("prediction_intervals", "permutation_second_minus_first_intervals", "baseline_shap_intervals"):
            for block in pair[panel]:
                for metric, interval in block["metrics"].items():
                    if panel != "prediction_intervals" and metric != GROUP_TO_INSPECT and not metric.startswith(GROUP_TO_INSPECT + "/"):
                        continue
                    interval_rows.append({
                        "first": pair["first"], "second": pair["second"], "panel": panel,
                        "block_days": block["block_days"], "occupied_blocks": block["blocks"],
                        "rows": block["rows"], "metric": metric, **interval,
                    })
    display(pd.DataFrame(interval_rows))
    print("Limitations:", *be2_report["limitations"], sep="\n- ")
else:
    print("Load a BE2 report before inspecting sensitivity and intervals.")

In [ ]:
import shutil

EXPORT_RESULTS = False
DOWNLOAD_ARCHIVE_IN_COLAB = False
EXPORT_LABEL = INSPECT_BE2_LABEL
if EXPORT_RESULTS:
    export_folder = RESULTS_DIR / EXPORT_LABEL
    if export_folder.parent.resolve() != RESULTS_DIR.resolve() or not export_folder.is_dir():
        raise ValueError("Select an existing run directory directly inside RESULTS_DIR")
    archive_base = STUDY_ROOT.parent / f"{EXPORT_LABEL}_artifacts"
    if archive_base.with_suffix(".zip").exists():
        raise FileExistsError("Archive already exists; choose a fresh export name")
    archive_path = shutil.make_archive(str(archive_base), "zip", root_dir=RESULTS_DIR, base_dir=EXPORT_LABEL)
    print("Exported:", archive_path)
    if DOWNLOAD_ARCHIVE_IN_COLAB:
        from google.colab import files
        files.download(archive_path)
else:
    print("Optional export disabled. In Colab, export each needed run before the runtime is discarded.")